# Day 068 — Exercise 1: OCR an Image

**What you'll build:** `ocr_image(img, ocr_fn=None, lang, config)` — extract text from a PIL Image using Tesseract, with mock injection for testing.

**Why it matters:** This is the Tesseract equivalent of `describe_image` from Day 67. The `ocr_fn=None` pattern makes every downstream document function testable without a running Tesseract installation — the same contract as `describe_fn` for vision LLMs.

In [ ]:
from PIL import Image, ImageDraw

_test_img = Image.new('RGB', (200, 50), color='white')
_draw = ImageDraw.Draw(_test_img)
_draw.text((10, 10), 'Hello OCR', fill='black')


## Task

Implement `ocr_image(img, ocr_fn=None, lang='eng', config='') -> str`:

- If `ocr_fn is not None`: call `ocr_fn(img)` and return the result
- Otherwise: `import pytesseract` and call `pytesseract.image_to_string(img, lang=lang, config=config)`
- Return the result string

All checks use a mock `ocr_fn` — Tesseract is not required.

## Your Implementation

In [ ]:
def ocr_image(img, ocr_fn=None, lang: str = 'eng', config: str = '') -> str:
    """Extract text from a PIL Image using Tesseract OCR.

    Args:
        img:    PIL Image to OCR
        ocr_fn: callable(img) -> str  (for testing without Tesseract)
        lang:   Tesseract language code, default 'eng'
        config: extra Tesseract flags, e.g. '--psm 6'
    Returns:
        Extracted text string
    """
    raise NotImplementedError


In [ ]:
def ocr_image(img, ocr_fn=None, lang: str = 'eng', config: str = '') -> str:
    if ocr_fn is not None:
        return ocr_fn(img)
    import pytesseract
    return pytesseract.image_to_string(img, lang=lang, config=config)


## Automated checks

In [ ]:
score, total = 0, 5
try:
    captured = {}
    def _mock(img_arg):
        captured['img'] = img_arg
        return 'Hello OCR'

    result = ocr_image(_test_img, ocr_fn=_mock)
    assert isinstance(result, str), f"Expected str, got {type(result)}"
    score += 1; print("\u2705 returns a string")

    assert result == 'Hello OCR', f"Expected 'Hello OCR', got {result!r}"
    score += 1; print("\u2705 returns the mock's text")

    assert captured.get('img') is _test_img, "mock should receive the img argument"
    score += 1; print("\u2705 mock receives the image")

    # lang and config are passed through (mock ignores them but they don't raise)
    result2 = ocr_image(_test_img, ocr_fn=lambda i: 'test', lang='fra', config='--psm 6')
    assert result2 == 'test'
    score += 1; print("\u2705 lang and config parameters accepted without error")

    # Default return is a string even for empty images
    result3 = ocr_image(Image.new('RGB',(50,20),'white'), ocr_fn=lambda i: '')
    assert isinstance(result3, str)
    score += 1; print("\u2705 empty-response mock returns empty string")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def ocr_image(img, ocr_fn=None, lang: str = 'eng', config: str = '') -> str:
    if ocr_fn is not None:
        return ocr_fn(img)
    import pytesseract
    return pytesseract.image_to_string(img, lang=lang, config=config)
```

**Why import inside the else branch?** If Tesseract is not installed, importing pytesseract raises an ImportError only when you actually try to OCR without a mock. The mock path never triggers the import. This lets the module load cleanly in environments where Tesseract is not available, while still failing loudly when you try to use the real OCR path without it.

</details>